# Pressure Ulcer QA — Inference

**Module:** 7146COMP — Advanced Topics in Deep Learning  
**Models:** Model B (bert-base-uncased, chunk=300) and Model D (Bio_ClinicalBERT, chunk=300)

This notebook loads both recommended fine-tuned extractive QA models and answers clinical questions about pressure ulcer prevention, staging, and management. Supply a clinical context passage and a question; each model returns a verbatim span from the context, or an empty string if the question is unanswerable given the context. Responses from Model B and Model D are shown side-by-side for direct comparison.

---

In [1]:
import torch
from transformers import BertForQuestionAnswering, BertTokenizerFast

MODEL_B_PATH = "./saved_model_300"          # Model B — bert-base-uncased
MODEL_D_PATH = "./saved_model_clinical_300" # Model D — Bio_ClinicalBERT
MAX_LENGTH   = 384

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Each model uses its own tokenizer — ClinicalBERT has a different vocabulary
tokenizer_b = BertTokenizerFast.from_pretrained(MODEL_B_PATH, local_files_only=True)
tokenizer_d = BertTokenizerFast.from_pretrained(MODEL_D_PATH, local_files_only=True)

model_b = BertForQuestionAnswering.from_pretrained(MODEL_B_PATH, local_files_only=True).to(device)
model_d = BertForQuestionAnswering.from_pretrained(MODEL_D_PATH, local_files_only=True).to(device)
model_b.eval()
model_d.eval()

print(f"Device   : {device}")
print(f"Model B  : {MODEL_B_PATH}")
print(f"Model D  : {MODEL_D_PATH}")

C:\Users\MSC1\anaconda3\envs\LLMCW\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device   : cuda
Model B  : ./saved_model_300
Model D  : ./saved_model_clinical_300


In [2]:
def predict_answer(question: str, context: str, model, tokenizer,
                   null_threshold: float = 0.0) -> str:
    """
    Return a verbatim span from `context` that answers `question`,
    or an empty string if the question is unanswerable.

    null_threshold:
        0.0  (default) — abstain whenever null score >= best span score
        positive       — require null to beat best span by this margin before abstaining
        negative       — abstain more conservatively
    """
    inputs = tokenizer(
        question, context,
        return_tensors="pt",
        max_length=MAX_LENGTH,
        truncation="only_second",
        return_offsets_mapping=True,
    )
    offset_mapping = inputs.pop("offset_mapping").squeeze(0)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    start_logits = outputs.start_logits.squeeze(0).cpu()
    end_logits   = outputs.end_logits.squeeze(0).cpu()
    null_score   = (start_logits[0] + end_logits[0]).item()
    seq_ids      = inputs.get("token_type_ids").squeeze(0).cpu().tolist()

    best_score, best_start, best_end = float("-inf"), 0, 0
    for s in range(len(start_logits)):
        for e in range(s, min(s + 50, len(end_logits))):
            if seq_ids[s] != 1:
                continue
            score = start_logits[s].item() + end_logits[e].item()
            if score > best_score:
                best_score, best_start, best_end = score, s, e

    if null_score - best_score >= null_threshold:
        return ""

    char_start = offset_mapping[best_start][0].item()
    char_end   = offset_mapping[best_end][1].item()
    return context[char_start:char_end]


def compare(question: str, context: str, null_threshold: float = 1.5):
    """Run both models on the same question/context and print a side-by-side result."""
    ans_b = predict_answer(question, context, model_b, tokenizer_b, null_threshold)
    ans_d = predict_answer(question, context, model_d, tokenizer_d, null_threshold)
    print(f"  Model B : {ans_b if ans_b else '(abstained)'}")
    print(f"  Model D : {ans_d if ans_d else '(abstained)'}")

---
## Clinical Examples

Five representative questions covering pressure ulcer staging, prevention, nutrition, risk assessment, and support surfaces. Each uses a context passage from the EPUAP/NPIAP/PPPIA guidelines.

### 1.5 Null Threshold

In [12]:
examples = [
    {
        "topic":    "Staging — Stage 3",
        "question": "What is visible in a Stage 3 pressure injury?",
        "context":  (
            "Pressure injuries are classified using the EPUAP/NPIAP staging system. "
            "Stage 1 is non-blanchable erythema of intact skin. "
            "Stage 2 involves partial thickness skin loss with exposed dermis. "
            "Stage 3 is full thickness skin loss in which adipose tissue is visible in the ulcer "
            "and granulation tissue and epibole are often present. "
            "Slough and eschar may be visible. The depth varies by anatomical location. "
            "Undermining and tunnelling may occur."
        ),
    },
    {
        "topic":    "Prevention — repositioning frequency",
        "question": "How often should high-risk patients be repositioned?",
        "context":  (
            "Repositioning is a cornerstone of pressure injury prevention. "
            "The EPUAP/NPIAP/PPPIA guidelines recommend that individuals at risk of pressure "
            "injuries who are bedbound should be repositioned at least every two hours. "
            "For individuals sitting in a chair or wheelchair, repositioning or "
            "pressure relief should be performed every hour. "
            "Frequency should be individualised based on skin assessment findings and "
            "the support surface in use."
        ),
    },
    {
        "topic":    "Nutrition — oral supplements",
        "question": "What nutritional supplements are recommended for patients with pressure injuries?",
        "context":  (
            "Nutrition plays a critical role in both the prevention and healing of pressure injuries. "
            "Patients who are malnourished or at nutritional risk should receive high-protein "
            "oral nutritional supplements in addition to their usual diet. "
            "Vitamin C and zinc supplementation may be considered for patients with documented "
            "deficiencies. Hydration status should be monitored and maintained."
        ),
    },
    {
        "topic":    "Risk assessment — Braden Scale",
        "question": "What does the Braden Scale measure?",
        "context":  (
            "The Braden Scale assesses pressure sore risk across six subscales: "
            "sensory perception, moisture, activity, mobility, nutrition, and friction and shear. "
            "Each subscale is scored 1–4 (except friction and shear, scored 1–3), "
            "giving a total range of 6–23. Lower scores indicate higher risk."
        ),
    },
    {
        "topic":    "Support surfaces — unanswerable",
        "question": "What is the recommended antibiotic for infected pressure ulcers?",
        "context":  (
            "Support surfaces are specialised devices for pressure redistribution, including "
            "mattresses, integrated bed systems, mattress overlays, seat cushions, and seat "
            "cushion overlays. "
            "Reactive support surfaces redistribute pressure only when loaded. "
            "Active support surfaces, including alternating pressure and low air loss systems, "
            "redistribute pressure without requiring the patient to change position. "
            "Selection should be based on individual patient risk assessment."
        ),
    },
]

NULL_THRESHOLD = 1.5

for ex in examples:
    print(f"[{ex['topic']}]")
    print(f"  Q: {ex['question']}")
    compare(ex["question"], ex["context"], null_threshold=NULL_THRESHOLD)
    print()

[Staging — Stage 3]
  Q: What is visible in a Stage 3 pressure injury?
  Model B : Slough and eschar may be visible
  Model D : (abstained)

[Prevention — repositioning frequency]
  Q: How often should high-risk patients be repositioned?
  Model B : at least every two hours
  Model D : at least every two hours

[Nutrition — oral supplements]
  Q: What nutritional supplements are recommended for patients with pressure injuries?
  Model B : high-protein oral nutritional supplements in addition to their usual diet
  Model D : (abstained)

[Risk assessment — Braden Scale]
  Q: What does the Braden Scale measure?
  Model B : (abstained)
  Model D : (abstained)

[Support surfaces — unanswerable]
  Q: What is the recommended antibiotic for infected pressure ulcers?
  Model B : (abstained)
  Model D : (abstained)



### 5.0 Null Threshold

In [13]:
examples = [
    {
        "topic":    "Staging — Stage 3",
        "question": "What is visible in a Stage 3 pressure injury?",
        "context":  (
            "Pressure injuries are classified using the EPUAP/NPIAP staging system. "
            "Stage 1 is non-blanchable erythema of intact skin. "
            "Stage 2 involves partial thickness skin loss with exposed dermis. "
            "Stage 3 is full thickness skin loss in which adipose tissue is visible in the ulcer "
            "and granulation tissue and epibole are often present. "
            "Slough and eschar may be visible. The depth varies by anatomical location. "
            "Undermining and tunnelling may occur."
        ),
    },
    {
        "topic":    "Prevention — repositioning frequency",
        "question": "How often should high-risk patients be repositioned?",
        "context":  (
            "Repositioning is a cornerstone of pressure injury prevention. "
            "The EPUAP/NPIAP/PPPIA guidelines recommend that individuals at risk of pressure "
            "injuries who are bedbound should be repositioned at least every two hours. "
            "For individuals sitting in a chair or wheelchair, repositioning or "
            "pressure relief should be performed every hour. "
            "Frequency should be individualised based on skin assessment findings and "
            "the support surface in use."
        ),
    },
    {
        "topic":    "Nutrition — oral supplements",
        "question": "What nutritional supplements are recommended for patients with pressure injuries?",
        "context":  (
            "Nutrition plays a critical role in both the prevention and healing of pressure injuries. "
            "Patients who are malnourished or at nutritional risk should receive high-protein "
            "oral nutritional supplements in addition to their usual diet. "
            "Vitamin C and zinc supplementation may be considered for patients with documented "
            "deficiencies. Hydration status should be monitored and maintained."
        ),
    },
    {
        "topic":    "Risk assessment — Braden Scale",
        "question": "What does the Braden Scale measure?",
        "context":  (
            "The Braden Scale assesses pressure sore risk across six subscales: "
            "sensory perception, moisture, activity, mobility, nutrition, and friction and shear. "
            "Each subscale is scored 1–4 (except friction and shear, scored 1–3), "
            "giving a total range of 6–23. Lower scores indicate higher risk."
        ),
    },
    {
        "topic":    "Support surfaces — unanswerable",
        "question": "What is the recommended antibiotic for infected pressure ulcers?",
        "context":  (
            "Support surfaces are specialised devices for pressure redistribution, including "
            "mattresses, integrated bed systems, mattress overlays, seat cushions, and seat "
            "cushion overlays. "
            "Reactive support surfaces redistribute pressure only when loaded. "
            "Active support surfaces, including alternating pressure and low air loss systems, "
            "redistribute pressure without requiring the patient to change position. "
            "Selection should be based on individual patient risk assessment."
        ),
    },
]

NULL_THRESHOLD = 5

for ex in examples:
    print(f"[{ex['topic']}]")
    print(f"  Q: {ex['question']}")
    compare(ex["question"], ex["context"], null_threshold=NULL_THRESHOLD)
    print()

[Staging — Stage 3]
  Q: What is visible in a Stage 3 pressure injury?
  Model B : Slough and eschar may be visible
  Model D : Slough and eschar

[Prevention — repositioning frequency]
  Q: How often should high-risk patients be repositioned?
  Model B : at least every two hours
  Model D : at least every two hours

[Nutrition — oral supplements]
  Q: What nutritional supplements are recommended for patients with pressure injuries?
  Model B : high-protein oral nutritional supplements in addition to their usual diet
  Model D : (abstained)

[Risk assessment — Braden Scale]
  Q: What does the Braden Scale measure?
  Model B : pressure sore risk across six subscales: sensory perception, moisture, activity, mobility, nutrition, and friction and shear
  Model D : sensory perception, moisture, activity, mobility, nutrition, and friction and shear

[Support surfaces — unanswerable]
  Q: What is the recommended antibiotic for infected pressure ulcers?
  Model B : (abstained)
  Model D : (abs

---
## Custom Query

Replace `QUESTION` and `CONTEXT` with your own clinical question and relevant guideline passage. Adjust `null_threshold` if needed:
- `0.0` — conservative default, abstains if null score ≥ best span score
- `+1.5` — deployment setting, reduces over-abstention on answerable questions

**Context guidance:** Keep the context focused on the specific clinical topic being asked about. This model was fine-tuned on single-topic passages (250–300 words). Providing a context that enumerates multiple similar entities (e.g., all four pressure injury stages in sequence) can cause **span misappropriation** — the model selects the highest-scoring span in the passage regardless of which entity the question specifies. Use a passage that covers the relevant topic in isolation for reliable results.

In [10]:
QUESTION = "What are the characteristics of a Stage 2 pressure injury?"

CONTEXT = """
Stage 2 pressure injuries involve partial thickness loss of skin with exposed dermis.
The wound bed is viable, pink or red and moist, and may present as an intact or ruptured
serum-filled blister. Adipose tissue and deeper tissues are not visible in a Stage 2 injury,
and no granulation tissue, slough or eschar is present.
"""

# Scan a range of thresholds to find where each model first commits to a span.
print(f"Question: {QUESTION}\n")
print(f"{'Threshold':>10}  {'Model B':40}  {'Model D'}")
print("-" * 100)
for t in [0.0, 1.0, 1.5, 2.0, 3.0, 5.0, 10.0]:
    ans_b = predict_answer(QUESTION, CONTEXT, model_b, tokenizer_b, null_threshold=t)
    ans_d = predict_answer(QUESTION, CONTEXT, model_d, tokenizer_d, null_threshold=t)
    b_str = ans_b if ans_b else "(abstained)"
    d_str = ans_d if ans_d else "(abstained)"
    print(f"  t={t:5.1f}   {b_str:40}  {d_str}")

Question: What are the characteristics of a Stage 2 pressure injury?

 Threshold  Model B                                   Model D
----------------------------------------------------------------------------------------------------
  t=  0.0   (abstained)                               (abstained)
  t=  1.0   (abstained)                               (abstained)
  t=  1.5   (abstained)                               (abstained)
  t=  2.0   (abstained)                               (abstained)
  t=  3.0   (abstained)                               (abstained)
  t=  5.0   (abstained)                               partial thickness loss of skin with exposed dermis
  t= 10.0   partial thickness loss of skin with exposed dermis  partial thickness loss of skin with exposed dermis


---
## Batch Inference

Pass a list of `(question, context)` pairs for batch evaluation. Useful for testing the model against a prepared question set.

In [9]:
import pandas as pd

def run_batch(pairs, null_threshold=5):
    """Run both models on a list of {question, context} dicts and return a comparison DataFrame."""
    records = []
    for item in pairs:
        q, ctx = item["question"], item["context"]
        ans_b = predict_answer(q, ctx, model_b, tokenizer_b, null_threshold)
        ans_d = predict_answer(q, ctx, model_d, tokenizer_d, null_threshold)
        records.append({
            "question":       q,
            "Model B answer": ans_b if ans_b else "(abstained)",
            "Model D answer": ans_d if ans_d else "(abstained)",
            "agree":          (bool(ans_b) == bool(ans_d)),
        })
    return pd.DataFrame(records)


batch_pairs = [{"question": ex["question"], "context": ex["context"]} for ex in examples]
df = run_batch(batch_pairs, null_threshold=5)
pd.set_option("display.max_colwidth", 60)
df

,question,Model B answer,Model D answer,agree
0,What is visible in a Stage 3 pressure injury?,Slough and eschar may be visible,Slough and eschar,True
1,How often should high-risk patients be repositioned?,at least every two hours,at least every two hours,True
2,What nutritional supplements are recommended for patient...,high-protein oral nutritional supplements in addition to...,(abstained),False
3,What does the Braden Scale measure?,pressure sore risk across six subscales: sensory percept...,"sensory perception, moisture, activity, mobility, nutrit...",True
4,What is the recommended antibiotic for infected pressure...,(abstained),(abstained),True


---
## Inference Results — Summary

The five clinical examples and the threshold scan below were run with `null_threshold=5.0`.

### Clinical examples

| Topic | Model B | Model D |
|---|---|---|
| Staging — Stage 3 | *"Slough and eschar may be visible"* ⚠ | *"Slough and eschar"* ⚠ |
| Prevention — repositioning frequency | *"at least every two hours"* ✓ | *"at least every two hours"* ✓ |
| Nutrition — oral supplements | *"high-protein oral nutritional supplements in addition to their usual diet"* ✓ | (abstained) |
| Risk assessment — Braden Scale | *"pressure sore risk across six subscales: sensory perception, moisture, activity, mobility, nutrition, and friction and shear"* ✓ | *"sensory perception, moisture, activity, mobility, nutrition, and friction and shear"* ✓ |
| Support surfaces — unanswerable | (abstained) ✓ | (abstained) ✓ |

### Key observations

**1. Both models now answer the Braden Scale question.**  
Raising the threshold from 1.5 to 5.0 unlocks this previously withheld answer for both models. Model D returns the cleaner span — just the six subscale names — while Model B includes the introductory clause `"pressure sore risk across six subscales:"`. Both are factually correct; Model D's span is more precisely bounded.

**2. Stage 3 remains off-target for both models.**  
Neither model returns the defining feature of Stage 3 (adipose tissue visibility). Model B returns `"Slough and eschar may be visible"` (a secondary feature); Model D returns the truncated `"Slough and eschar"` — even less precise. The context contains the correct span (`"adipose tissue is visible in the ulcer"`), but both models assign higher confidence to the slough/eschar mention. This is a consistent weakness across both thresholds.

**3. Model D still abstains on the nutrition question at threshold 5.0.**  
Despite the raised threshold, Model D's null score on this question exceeds 5.0 above the best span score, indicating very low confidence. Model B continues to return the correct span. This is the one question where Model B clearly outperforms Model D regardless of threshold.

**4. Unanswerable question still correctly rejected by both.**  
The antibiotic question continues to produce abstentions from both models even at threshold 5.0, confirming that null detection remains reliable at this setting.

**5. Agreement on repositioning is unchanged.**  
Both models continue to return the identical span `"at least every two hours"` — the most unambiguous fact in the set.

### Threshold scan — Stage 2 characteristics

Both models abstain for all thresholds up to `t=3.0`.  
Model D commits at **`t=5.0`** returning `"partial thickness loss of skin with exposed dermis"` — a precise, clinically correct span.  
Model B requires **`t=10.0`** to commit, returning the same span.  
Model D's earlier commit reflects Bio_ClinicalBERT's stronger prior on dermis-level skin loss terminology.

### Threshold guidance

| `null_threshold` | Behaviour |
|---|---|
| 0.0 | Both models abstain on almost everything. |
| 1.5 | Answers clear numerical facts; abstains on list-type and staging questions. |
| **5.0** | **Current setting.** Recovers Braden Scale and Stage 3 answers. Stage 3 spans remain imprecise; Model D still abstains on nutrition. |
| 10.0+ | Both models answer most questions; false span risk increases. Not recommended for clinical deployment. |

---
## Model Comparison — Which Model Performs Better?

### Results at null_threshold = 5.0

| Metric | Model B (bert-base-uncased) | Model D (Bio_ClinicalBERT) |
|---|---|---|
| Questions answered (of 5) | 4 | 3 |
| Unanswerable correctly rejected | ✓ | ✓ |
| Stage 3 span | `"Slough and eschar may be visible"` ⚠ | `"Slough and eschar"` ⚠ |
| Repositioning span | `"at least every two hours"` ✓ | `"at least every two hours"` ✓ |
| Nutrition span | `"high-protein oral nutritional supplements in addition to their usual diet"` ✓ | (abstained) |
| Braden Scale span | `"pressure sore risk across six subscales: sensory perception..."` ✓ | `"sensory perception, moisture, activity, mobility, nutrition, and friction and shear"` ✓ |
| Stage 2 threshold scan — first commit | t = 10.0 | t = 5.0 |

### Interpretation

At `null_threshold=5.0`, Model B answers 4 of 5 questions and Model D answers 3 of 5. The key differences are:

**Stage 3 — both models are wrong.**  
The correct defining feature of Stage 3 is adipose tissue visibility. Both models select the slough/eschar mention instead — Model B returns a full clause, Model D returns a truncated fragment (`"Slough and eschar"`). Neither answer is acceptable for clinical use on this question at this threshold. This reflects a shared limitation: the slough/eschar tokens attract higher logit mass despite not being the primary distinguishing feature.

**Braden Scale — Model D's span is cleaner.**  
Both models now answer (threshold 5.0 unblocks this question from the 1.5 abstention). Model D returns only the six subscale names, which is the precise answer. Model B prepends `"pressure sore risk across six subscales:"` — factually correct but slightly over-spans. Model D wins on span precision here.

**Nutrition — Model B's only clear win.**  
Model D abstains on the nutrition question even at threshold 5.0, meaning its null score beats the best span by more than 5 points. Model B returns the correct span. This is the one case where Model B is unambiguously better regardless of threshold setting.

**Unanswerable — both reliable.**  
Both models continue to abstain on the antibiotic question. Null detection holds at threshold 5.0.

### Updated evaluation context

| Metric | Model B | Model D | Winner |
|---|---|---|---|
| BERTScore F1 (answerable only) | 0.9242 | **0.9285** | D |
| False span rate | 9 | **7** | D |
| Null detection rate | 93.7% | **95.1%** | D |
| G-Eval Accuracy /5 | 4.05 | **4.10** | D |
| G-Eval Clarity /5 | 4.15 | **4.30** | D |
| Bias rate | 0.11 | **0.00** | D |
| Over-abstention rate | **30.5%** | 33.8% | B |
| Prompt compliance | 0.9854 | **0.9887** | D |

### Conclusion

**Model D (Bio_ClinicalBERT-300) remains the stronger model overall.** At `null_threshold=5.0`, it now recovers the Braden Scale answer with a cleaner, more precisely bounded span than Model B. Its evaluation metrics continue to show better span quality, fewer false answers, zero bias, and superior null detection.

The two remaining weaknesses at this threshold are shared with Model B (Stage 3 off-target span) or isolated to a single question (nutrition abstention). For clinical deployment, Model D at `null_threshold=5.0` is the recommended configuration. If the nutrition question is a high-priority use case, consider raising the threshold further for Model D on that question class specifically, or using Model B as a fallback for nutrition-related queries only.

---
## Next Steps — Retrieval-Augmented Generation (RAG)

The inference results above highlight a fundamental limitation of the current pipeline: the model depends entirely on the user supplying a relevant, well-formed context passage. In practice, a Mersey Care NHS clinician cannot be expected to locate and paste the correct EPUAP guideline paragraph before asking a question — this removes the core value of the system.

Retrieval-Augmented Generation (RAG) addresses this directly. In a RAG pipeline, a retriever component — typically a dense bi-encoder such as `sentence-transformers/all-MiniLM-L6-v2` — encodes the full guideline corpus into a vector index at indexing time. At query time, the clinician's question is encoded and the top-*k* most semantically similar passages are retrieved automatically. These passages are then passed as context to `predict_answer`, replacing the manual context step entirely.

This would resolve both failure modes identified above: the disambiguation problem (the retriever selects only the relevant staging passage, not all four stages at once) and the distribution-shift problem (retrieved passages are verbatim guideline text, matching the training distribution).

The theory, architecture, and implementation of a RAG pipeline over the pressure ulcer corpus will be analysed and demonstrated in the next notebook.